# Distributed Agent System Test Notebook

This notebook tests the distributed proxy/dispatcher architecture with multiple service agent clusters.

## Prerequisites
Before running this notebook, start the distributed services:
```bash
docker-compose -f docker-compose.distributed.yml up -d
```

## Test Scenarios
1. Proxy Service Health
2. Service Discovery
3. Direct Service Communication
4. Cross-Service Agent Communication
5. Health Check and Fault Tolerance
6. Multi-Service Collaborative Analysis


In [1]:
# Setup and Configuration
import requests
import json
import time

# Service URLs
PROXY_URL = "http://localhost:8000"
SERVICE_A_URL = "http://localhost:8001"  # payment-service
SERVICE_B_URL = "http://localhost:8002"  # order-service

def pretty_print(data):
    """Pretty print JSON data."""
    print(json.dumps(data, indent=2, default=str))

print("Configuration loaded!")
print(f"Proxy URL: {PROXY_URL}")
print(f"Service A URL: {SERVICE_A_URL}")
print(f"Service B URL: {SERVICE_B_URL}")


Configuration loaded!
Proxy URL: http://localhost:8000
Service A URL: http://localhost:8001
Service B URL: http://localhost:8002


In [2]:
# Test 1: Proxy Service Health Check
print("=" * 60)
print("TEST 1: Proxy Service Health")
print("=" * 60)

try:
    response = requests.get(f"{PROXY_URL}/health", timeout=10)
    print(f"\nStatus Code: {response.status_code}")
    print("\nProxy Health Response:")
    pretty_print(response.json())
    
    if response.status_code == 200:
        print("\n✅ Proxy service is healthy!")
    else:
        print("\n❌ Proxy service returned non-200 status")
        
except requests.exceptions.ConnectionError:
    print("\n❌ Cannot connect to proxy service")
    print("Make sure you've started the services with:")
    print("docker-compose -f docker-compose.distributed.yml up -d")


TEST 1: Proxy Service Health

Status Code: 200

Proxy Health Response:
{
  "status": "healthy",
  "service": "proxy",
  "registered_services": 2,
  "healthy_services": 2,
  "timestamp": "2025-12-11T15:39:15.900672"
}

✅ Proxy service is healthy!


In [3]:
# Test 2: Service Discovery
print("=" * 60)
print("TEST 2: Service Discovery")
print("=" * 60)

# List all services
print("\n--- All Registered Services ---")
try:
    response = requests.get(f"{PROXY_URL}/services", timeout=10)
    print(f"Status Code: {response.status_code}")
    data = response.json()
    pretty_print(data)
    print(f"\nTotal services: {data.get('count', 0)}")
except Exception as e:
    print(f"Error: {e}")

# Discover by capability
print("\n--- Discover Services with 'payment' capability ---")
try:
    response = requests.get(f"{PROXY_URL}/discover", params={"capability": "payment"}, timeout=10)
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

print("\n--- Discover Services with 'order' capability ---")
try:
    response = requests.get(f"{PROXY_URL}/discover", params={"capability": "order"}, timeout=10)
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")


TEST 2: Service Discovery

--- All Registered Services ---
Status Code: 200
{
  "services": [
    {
      "name": "order-service",
      "url": "http://service-b-api:8000",
      "capabilities": [
        "order",
        "fulfillment",
        "inventory",
        "shipping"
      ],
      "version": "1.0.0",
      "description": "Order Service Agent - Analyzes order processing logs and errors",
      "status": "healthy",
      "registered_at": "2025-12-11T15:37:07.030215",
      "last_heartbeat": "2025-12-11T15:39:07.109559",
      "heartbeat_count": 5
    },
    {
      "name": "payment-service",
      "url": "http://service-a-api:8000",
      "capabilities": [
        "payment",
        "billing",
        "transaction",
        "financial"
      ],
      "version": "1.0.0",
      "description": "Payment Service Agent - Analyzes payment-related logs and errors",
      "status": "healthy",
      "registered_at": "2025-12-11T15:37:07.079451",
      "last_heartbeat": "2025-12-11T15:39:

In [4]:
# Test 3: Direct Service Communication
print("=" * 60)
print("TEST 3: Direct Service Communication")
print("=" * 60)

# Check Service A health and info
print("\n--- Service A (payment-service) Health ---")
try:
    response = requests.get(f"{SERVICE_A_URL}/health", timeout=10)
    print(f"Status: {response.status_code}")
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

print("\n--- Service A Info ---")
try:
    response = requests.get(f"{SERVICE_A_URL}/api/service-info", timeout=10)
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

# Check Service B health and info
print("\n--- Service B (order-service) Health ---")
try:
    response = requests.get(f"{SERVICE_B_URL}/health", timeout=10)
    print(f"Status: {response.status_code}")
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

print("\n--- Service B Info ---")
try:
    response = requests.get(f"{SERVICE_B_URL}/api/service-info", timeout=10)
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")


TEST 3: Direct Service Communication

--- Service A (payment-service) Health ---
Status: 200
{
  "capabilities": [
    "payment",
    "billing",
    "transaction",
    "financial"
  ],
  "proxy_configured": true,
  "service_name": "payment-service",
  "status": "ok"
}

--- Service A Info ---
{
  "capabilities": [
    "payment",
    "billing",
    "transaction",
    "financial"
  ],
  "description": "Payment Service Agent - Analyzes payment-related logs and errors",
  "name": "payment-service",
  "proxy_enabled": true,
  "proxy_url": "http://proxy:8000",
  "url": "http://service-a-api:8000",
  "version": "1.0.0"
}

--- Service B (order-service) Health ---
Status: 200
{
  "capabilities": [
    "order",
    "fulfillment",
    "inventory",
    "shipping"
  ],
  "proxy_configured": true,
  "service_name": "order-service",
  "status": "ok"
}

--- Service B Info ---
{
  "capabilities": [
    "order",
    "fulfillment",
    "inventory",
    "shipping"
  ],
  "description": "Order Service Agent

In [5]:
# Test 4: Cross-Service Agent Communication
print("=" * 60)
print("TEST 4: Cross-Service Agent Communication")
print("=" * 60)

# Service A discovers other services
print("\n--- Service A Discovers Other Services ---")
try:
    response = requests.get(f"{SERVICE_A_URL}/api/discover-services", timeout=10)
    print("Services discovered by Service A:")
    pretty_print(response.json())
except Exception as e:
    print(f"Error: {e}")

# Service A queries Service B
print("\n--- Service A Queries Service B for Analysis ---")
test_error = """
2025-01-14 10:30:15,123 [ERROR] [payment-service]
Payment processing failed for order #12345
java.sql.SQLException: Connection timeout to payment gateway
  at com.payment.Gateway.connect(Gateway.java:87)
  at com.payment.Processor.processPayment(Processor.java:145)
"""

try:
    response = requests.post(
        f"{SERVICE_A_URL}/api/query-service/order-service",
        json={"query": test_error},
        timeout=120
    )
    print(f"Status: {response.status_code}")
    result = response.json()
    pretty_print(result)
except Exception as e:
    print(f"Error: {e}")


TEST 4: Cross-Service Agent Communication

--- Service A Discovers Other Services ---
Services discovered by Service A:
{
  "count": 2,
  "filter": null,
  "services": [
    {
      "capabilities": [
        "order",
        "fulfillment",
        "inventory",
        "shipping"
      ],
      "description": "Order Service Agent - Analyzes order processing logs and errors",
      "heartbeat_count": 5,
      "last_heartbeat": "2025-12-11T15:39:07.109559",
      "name": "order-service",
      "registered_at": "2025-12-11T15:37:07.030215",
      "status": "healthy",
      "url": "http://service-b-api:8000",
      "version": "1.0.0"
    },
    {
      "capabilities": [
        "payment",
        "billing",
        "transaction",
        "financial"
      ],
      "description": "Payment Service Agent - Analyzes payment-related logs and errors",
      "heartbeat_count": 5,
      "last_heartbeat": "2025-12-11T15:39:07.134888",
      "name": "payment-service",
      "registered_at": "2025-12-

In [6]:
# Test 5: Multi-Service Collaborative Analysis
print("=" * 60)
print("TEST 5: Multi-Service Collaborative Analysis")
print("=" * 60)

# Complex error that spans multiple services
complex_error = """
2025-01-14 12:15:42,123 [ERROR] [payment-service]
Critical payment processing failure detected.

Transaction Details:
- Order ID: #98765
- Amount: $1,250.00
- Customer: customer_456

Error Stack:
javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException
Unable to acquire JDBC Connection
  at org.hibernate.internal.ExceptionConverterImpl.convert(ExceptionConverterImpl.java:154)
Caused by: java.sql.SQLTransientConnectionException: HikariPool-1 - Connection timeout after 30000ms.

This error may be affecting order fulfillment and inventory management.
"""

print("\nSubmitting complex error for multi-service analysis...")
print("\n" + "-" * 40)
print("Error being analyzed:")
print(complex_error[:400] + "...")
print("-" * 40)

# Query Service A for comprehensive analysis
print("\n--- Querying payment-service for comprehensive analysis ---")
print("(This may take a minute as the agent queries multiple services)")

try:
    start_time = time.time()
    response = requests.post(
        f"{SERVICE_A_URL}/api/query",
        json={
            "query": f"""
Analyze this critical payment error and provide a comprehensive report.
Use cross-service analysis to understand the full impact.

Error Log:
{complex_error}
"""
        },
        timeout=180
    )
    duration = time.time() - start_time
    
    print(f"\nStatus: {response.status_code}")
    print(f"Duration: {duration:.2f} seconds")
    
    if response.status_code == 200:
        result = response.json()
        print("\n" + "=" * 60)
        print("COMPREHENSIVE ANALYSIS REPORT")
        print("=" * 60)
        print(result.get('reply', 'No reply')[:3000])
    else:
        print(f"Error: {response.text}")
        
except Exception as e:
    print(f"Error: {e}")


TEST 5: Multi-Service Collaborative Analysis

Submitting complex error for multi-service analysis...

----------------------------------------
Error being analyzed:

2025-01-14 12:15:42,123 [ERROR] [payment-service]
Critical payment processing failure detected.

Transaction Details:
- Order ID: #98765
- Amount: $1,250.00
- Customer: customer_456

Error Stack:
javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException
Unable to acquire JDBC Connection
  at org.hibernate.internal.ExceptionConverterImpl.convert(ExceptionConverterImpl...
----------------------------------------

--- Querying payment-service for comprehensive analysis ---
(This may take a minute as the agent queries multiple services)

Status: 200
Duration: 104.84 seconds

COMPREHENSIVE ANALYSIS REPORT
LOG ANALYSIS REPORT

Original Log:

Analyze this critical payment error and provide a comprehensive report.
Use cross-service analysis to understand the full impact.

Error Log:

2025-01-14 12:15:

In [ ]:
# Test 7: Verify Seeded Data in PostgreSQL
print("=" * 60)
print("TEST 7: Verify Seeded Data in PostgreSQL")
print("=" * 60)

import psycopg2

def query_postgres(host, port, dbname, query):
    """Query PostgreSQL and return results."""
    try:
        conn = psycopg2.connect(
            host=host, port=port, dbname=dbname,
            user="logs_user", password="logs_pass"
        )
        cur = conn.cursor()
        cur.execute(query)
        results = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
        conn.close()
        return columns, results
    except Exception as e:
        return None, str(e)

# Check Service A PostgreSQL
print("\n--- Service A (payment-service) PostgreSQL ---")
try:
    cols, results = query_postgres("localhost", 5433, "service_a_logs", 
                                    "SELECT level, raw->>'message' as message FROM logs LIMIT 5")
    if cols:
        print(f"Found {len(results)} sample logs:")
        for row in results:
            print(f"  [{row[0]}] {row[1][:60]}...")
except Exception as e:
    print(f"Error: {e}")

# Check Service B PostgreSQL
print("\n--- Service B (order-service) PostgreSQL ---")
try:
    cols, results = query_postgres("localhost", 5434, "service_b_logs",
                                    "SELECT level, raw->>'message' as message FROM logs LIMIT 5")
    if cols:
        print(f"Found {len(results)} sample logs:")
        for row in results:
            print(f"  [{row[0]}] {row[1][:60]}...")
except Exception as e:
    print(f"Error: {e}")


In [ ]:
# Test 8: Verify Seeded Data in Qdrant
print("=" * 60)
print("TEST 8: Verify Seeded Data in Qdrant")
print("=" * 60)

from qdrant_client import QdrantClient

def check_qdrant(url, collection_name):
    """Check Qdrant collection status."""
    try:
        client = QdrantClient(url=url, timeout=10)
        info = client.get_collection(collection_name)
        return info.points_count
    except Exception as e:
        return f"Error: {e}"

# Check Service A Qdrant
print("\n--- Service A (payment-service) Qdrant ---")
count = check_qdrant("http://localhost:6333", "log_fixes")
print(f"  Collection 'log_fixes': {count} error-fix pairs")

# Check Service B Qdrant
print("\n--- Service B (order-service) Qdrant ---")
count = check_qdrant("http://localhost:6334", "log_fixes")
print(f"  Collection 'log_fixes': {count} error-fix pairs")

# Sample search to verify
print("\n--- Sample RAG Search Test ---")
try:
    from openai import OpenAI
    import os
    
    client = QdrantClient(url="http://localhost:6333", timeout=10)
    openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    
    # Generate embedding for test query
    query = "payment gateway timeout"
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )
    query_vector = response.data[0].embedding
    
    # Search
    results = client.query_points(
        collection_name="log_fixes",
        query=query_vector,
        limit=2,
        with_payload=True
    )
    
    print(f"Query: '{query}'")
    print(f"Found {len(results.points)} results:")
    for i, point in enumerate(results.points, 1):
        error = point.payload.get('error', 'N/A')[:50]
        print(f"  {i}. Score: {point.score:.4f} | Error: {error}...")
except Exception as e:
    print(f"Search test error: {e}")


In [8]:
# Test 9: Realistic Cross-Service Demo Scenario
print("=" * 60)
print("TEST 9: Realistic Cross-Service Demo Scenario")
print("=" * 60)

# This scenario simulates a real-world cascading failure:
# - Payment service experiences database connection pool exhaustion
# - This causes payment timeouts
# - Which blocks order fulfillment in the order service

demo_error = """
2025-01-14 08:30:15,456 [ERROR] [payment-service]
Critical payment processing failure affecting multiple orders.

Stacktrace:
javax.persistence.PersistenceException: org.hibernate.exception.JDBCConnectionException
Unable to acquire JDBC Connection
  at org.hibernate.internal.ExceptionConverterImpl.convert(ExceptionConverterImpl.java:154)
  at org.hibernate.internal.SessionImpl.doFlush(SessionImpl.java:1483)
Caused by: java.sql.SQLTransientConnectionException: HikariPool-1 - Connection is not available, 
request timed out after 30000ms.
  at com.zaxxer.hikari.pool.HikariPool.createTimeoutException(HikariPool.java:695)

Affected transactions:
- Order ORD-10055: Payment pending for 45 minutes
- Order ORD-10056: Payment pending for 42 minutes  
- Order ORD-10057: Payment pending for 40 minutes

This is causing cascading failures in the order fulfillment pipeline.
Orders are stuck in PENDING_PAYMENT state and cannot be shipped.

Please investigate cross-service impact and provide comprehensive analysis.
"""

print("\nDemo Scenario: Payment Service Database Connection Pool Exhaustion")
print("-" * 60)
print("This error will trigger:")
print("1. Payment service internal knowledge search (local DB + RAG)")
print("2. Cross-service discovery via proxy")
print("3. Query to order-service for impact assessment")
print("4. Comprehensive multi-service analysis report")
print("-" * 60)
print("\nSubmitting to payment-service for analysis...")
print("(This may take 1-2 minutes as the agent performs cross-service analysis)")

try:
    start_time = time.time()
    response = requests.post(
        f"{SERVICE_A_URL}/api/query",
        json={
            "query": f"""
Analyze this critical error and provide a comprehensive cross-service analysis.

Instructions:
1. First, analyze the error locally using your internal knowledge
2. Discover other services in the system
3. Query the order-service to understand the downstream impact
4. Synthesize all information into a comprehensive report

Error Log:
{demo_error}
"""
        },
        timeout=300
    )
    duration = time.time() - start_time
    
    print(f"\nStatus: {response.status_code}")
    print(f"Analysis Duration: {duration:.1f} seconds")
    
    if response.status_code == 200:
        result = response.json()
        print("\n" + "=" * 60)
        print("CROSS-SERVICE ANALYSIS REPORT")
        print("=" * 60)
        report = result.get('reply', 'No reply')
        # Print report in chunks for readability
        print(report[:4000])
        if len(report) > 4000:
            print("\n... [Report truncated] ...")
    else:
        print(f"Error: {response.text}")
        
except requests.exceptions.Timeout:
    print("Request timed out - analysis is taking longer than expected")
except Exception as e:
    print(f"Error: {e}")


TEST 9: Realistic Cross-Service Demo Scenario

Demo Scenario: Payment Service Database Connection Pool Exhaustion
------------------------------------------------------------
This error will trigger:
1. Payment service internal knowledge search (local DB + RAG)
2. Cross-service discovery via proxy
3. Query to order-service for impact assessment
4. Comprehensive multi-service analysis report
------------------------------------------------------------

Submitting to payment-service for analysis...
(This may take 1-2 minutes as the agent performs cross-service analysis)

Status: 200
Analysis Duration: 146.4 seconds

CROSS-SERVICE ANALYSIS REPORT
LOG ANALYSIS REPORT

Original Log:

Analyze this critical error and provide a comprehensive cross-service analysis.

Instructions:
1. First, analyze the error locally using your internal knowledge
2. Discover other services in the system
3. Query the order-service to understand the downstream impact
4. Synthesize all information into a comprehens

In [7]:
# Test 6: Summary and System Status
print("=" * 60)
print("DISTRIBUTED SYSTEM TEST SUMMARY")
print("=" * 60)

tests_passed = 0
tests_failed = 0

# Check proxy
try:
    r = requests.get(f"{PROXY_URL}/health", timeout=5)
    if r.status_code == 200:
        print("✅ Proxy: Healthy")
        tests_passed += 1
    else:
        print("❌ Proxy: Unhealthy")
        tests_failed += 1
except:
    print("❌ Proxy: Unavailable")
    tests_failed += 1

# Check Service A
try:
    r = requests.get(f"{SERVICE_A_URL}/health", timeout=5)
    if r.status_code == 200:
        print("✅ Service A (payment-service): Healthy")
        tests_passed += 1
    else:
        print("❌ Service A: Unhealthy")
        tests_failed += 1
except:
    print("❌ Service A: Unavailable")
    tests_failed += 1

# Check Service B
try:
    r = requests.get(f"{SERVICE_B_URL}/health", timeout=5)
    if r.status_code == 200:
        print("✅ Service B (order-service): Healthy")
        tests_passed += 1
    else:
        print("❌ Service B: Unhealthy")
        tests_failed += 1
except:
    print("❌ Service B: Unavailable")
    tests_failed += 1

# Registered services
try:
    r = requests.get(f"{PROXY_URL}/services", timeout=5)
    data = r.json()
    print(f"\n📊 Registered Services: {data.get('count', 0)}")
    for svc in data.get('services', []):
        status_icon = "✅" if svc['status'] == 'healthy' else "⚠️"
        print(f"   {status_icon} {svc['name']}: {svc.get('capabilities', [])}")
except:
    pass

print(f"\n--- Results ---")
print(f"Tests Passed: {tests_passed}")
print(f"Tests Failed: {tests_failed}")

print("\n" + "=" * 60)
print("CLEANUP INSTRUCTIONS")
print("=" * 60)
print("""
To stop all distributed services:
  docker-compose -f docker-compose.distributed.yml down

To stop and remove all data:
  docker-compose -f docker-compose.distributed.yml down -v

To view logs:
  docker-compose -f docker-compose.distributed.yml logs -f
""")


DISTRIBUTED SYSTEM TEST SUMMARY
✅ Proxy: Healthy
✅ Service A (payment-service): Healthy
✅ Service B (order-service): Healthy

📊 Registered Services: 2
   ✅ order-service: ['order', 'fulfillment', 'inventory', 'shipping']
   ✅ payment-service: ['payment', 'billing', 'transaction', 'financial']

--- Results ---
Tests Passed: 3
Tests Failed: 0

CLEANUP INSTRUCTIONS

To stop all distributed services:
  docker-compose -f docker-compose.distributed.yml down

To stop and remove all data:
  docker-compose -f docker-compose.distributed.yml down -v

To view logs:
  docker-compose -f docker-compose.distributed.yml logs -f

